In [ ]:
#| hide
from vpseasy.core import *
import os, tempfile
from pathlib import Path

/Users/71293/code/personal/orgs/vpseasy/.venv/lib/python3.13/site-packages/fastprogress/fastprogress.py:171: UserWarning: Couldn't import ipython display functions, progress bar will use console behavior
  warn("Couldn't import ipython display functions, progress bar will use console behavior")


# vpseasy

> Provision VPSes, test locally with Multipass, deploy Docker Compose apps — all from Python.

```sh
pip install vpseasy
```

`vpseasy` turns the full VPS lifecycle into a handful of function calls:

| Step | Function | What it does |
|------|----------|--------------|
| 1 | `vps_init()` / `multi_init()` | Generate cloud-init YAML |
| 2 | `Multipass.launch()` | Spin up a local Ubuntu VM to test your setup |
| 3 | `Hetzner.create()` | Provision a real VPS on Hetzner Cloud |
| 4 | `wait_ssh()` + `deploy()` | Wait for SSH, rsync your app, `docker compose up` |

## Install

```sh
pip install vpseasy
```

In [ ]:
pub_keys = load_pub_keys()
print(f'{len(pub_keys)} key(s) found')
if pub_keys: print(pub_keys[0][:3] + '...')

4 key(s) found
ssh...


## Cloud-init

`multi_init()` — local Multipass VMs (no UFW). `vps_init()` — production (UFW, fail2ban, Docker).

In [ ]:
_vm = 'testvm'
mi = multi_init(_vm, docker=False)   # docker=False: skip install+reboot, much faster for local testing
print(mi.yaml)

In [ ]:
ci = vps_init('demo-prod', pub_keys)
print(ci.yaml)

#cloud-config
hostname: demo-prod
preserve_hostname: false
packages:
- curl
- fail2ban
- unattended-upgrades
package_update: true
package_upgrade: true
disable_root: true
ssh_pwauth: false
users:
- name: deploy
  groups:
  - sudo
  shell: /bin/bash
  sudo:
  - ALL=(ALL) NOPASSWD:ALL
  ssh_authorized_keys:
  - ssh-ed25519 AAAAC3NzaC1lZDI1NTE5AAAAIAcfGCEzt9TJzVOBmlzU4N8LvLKQxUQQ/mIikwArFO1K karthikrajgopal.laxminaarayanan@bain.com
  - ssh-rsa AAAAB3NzaC1yc2EAAAADAQABAAACAQDM9kAjDCitQvl6oMLab4yK7LgylUV9lo/M/U8uqi5r8LPbo/qia7Oaj+Qo2LSs+Hsesr/mdKNjFle5DRb91k/Ns4GX2V8QyBcKwBK0Davjmj57X+4ZfnmC4HjZ/gzyYe1hV4Vjy4IBE/JSArSP3hI8tCAgN00tNLxJ5AJnYzdgfKpVa+zI54cdrJTPhko12mEyOih62xOWeHT16Y7jGPIaOzPo2YTFM+omwEm7TfyxuRLxaDN0d/ZxlKPi/+vBcuAOItrjA6DJ7lnYwk3cXubCKgHP3uc0MeBBX74S58zpoHlAp6XdePKOoBwam1/aYm+7zq+9GJyDGV25iD9bDwSn+0oNexJFRgQxwFdIwVUk4Iyq6OocUNLX8vrI1Qr6kXIchDVS922LGf+Z5aai89wqaxLLB+U4+dNTzb6zKMtQSMdGrgFZt0N5j/aMuRE5rkfWoeiCT8DmekuxA6NDaF76CYgcIsEaOsCTuNjwrpWBQnQ82r20tfegagT3Y38xBp9PLbFHbM45Hkj

## Local testing with Multipass

Requires [Multipass](https://multipass.run) installed. Pass `cloud_init=mi` directly to `mp.launch()`. Use `docker=True` in `multi_init()` if your app needs Docker pre-installed (adds ~2 min for install + reboot).

In [ ]:
#| eval: False
mp = Multipass()
mp.rm(_vm,purge=True)   # idempotent cleanup
vm = mp.launch(_vm, image='24.04', cpus=1, memory='1G', disk='10G', cloud_init=mi)
ip = mp.ip(vm.name)
print(f'VM at {ip}, key: {vm.key}')
deploy_mp(_vm, src='./myapp')
mp.rm(_vm)

Creating testvm  Configuring testvm  Starting testvm  Waiting for initialization to complete  

launch failed: The following errors occurred:
timed out waiting for initialization to complete


CalledProcessError: Command '['multipass', 'launch', '24.04', '-n', 'testvm', '-c', '1', '-m', '1G', '-d', '10G', '--cloud-init', '-']' returned non-zero exit status 2.

## Provision on Hetzner

Set `HCLOUD_TOKEN` in your environment. `vps_init()` auto-generates an SSH key pair when `pub_keys=None`.

In [ ]:
#| eval: False
hz = Hetzner()                              # reads HCLOUD_TOKEN
ci = vps_init('myapp-prod', pub_keys)
svr = hz.create('myapp-prod', cloud_init=ci, ssh_keys=hz.key_names(), location='hel1')
print(f'Provisioning at {svr.ip}')

## Deploy

`wait_ssh()` blocks until SSH is up. `deploy()` rsyncs your Compose stack and brings it up.

In [ ]:
#| eval: False
wait_ssh(svr.ip, tout=300)
assert chk_cloud_init(svr.ip) == 'done'
assert chk_docker(svr.ip)
deploy('./myapp', svr.ip)

## Install agent skill

Copies `SKILL.md` to `.agents/skills/vpseasy/` (project-local) and `~/.claude/skills/vpseasy/` (global Claude Code).

In [ ]:
mv_skill_md(dry_run=True)   # preview; pass dry_run=False to actually install

## API reference

| Symbol | Description |
|--------|-------------|
| `load_pub_keys(paths=None)` | Read `~/.ssh/id_*.pub` → list of strings |
| `gen_key(slug, key_dir=None)` | Generate ed25519 pair → `AttrDict(key, pub, pub_str)` |
| `multi_init(hostname, pub_keys, ...)` | Multipass cloud-init YAML → `AttrDict(yaml, key)` |
| `vps_init(hostname, pub_keys, ...)` | Production cloud-init YAML → `AttrDict(yaml, key)` |
| `Multipass` | Launch / list / exec / delete local Ubuntu VMs |
| `deploy_mp(name, src, path, build)` | Sync dir + `docker compose up` in Multipass VM |
| `Hetzner` | Create / list / delete Hetzner Cloud servers |
| `wait_ssh(host, u, k, tout)` | Poll until SSH accepts connections |
| `chk_cloud_init(host, u, k)` | Return `cloud-init status` string |
| `chk_docker(host, u, k)` | Verify Docker daemon running |
| `run_ssh(host, *cmds, ...)` | Run commands over SSH |
| `sync(host, src, path, ...)` | Rsync local dir to remote |
| `deploy(host, src, path, ...)` | `sync` + `docker compose up -d` |
| `mv_skill_md(dry_run, dir)` | Install agent SKILL.md |